# **[Tutorial:](https://github.com/yashizhang/esm/tree/main/cookbook/tutorials) Understanding Proteins with SAE Features**

SAE (Sparse Autoencoder) features provide an interpretable view into ESMC’s internal representations, helping reveal what patterns the model associates with different regions of a protein sequence. Individual features often correspond to recognizable biological concepts such as structural motifs, binding sites, biophysical properties, or evolutionary/family signals.

## **What are SAE features?**

When ESMC processes a protein sequence, it represents each residue as a dense embedding vector. These dense vectors encode rich biological information, but are difficult to interpret directly because individual embedding dimensions can reflect multiple unrelated concepts.

Sparse Autoencoders (SAEs) are interpretability models trained to translate dense embeddings into a much larger but sparse feature space.

In this tutorial we use a Top-k SAE trained for ESMC. Top-K means that for each residue, only the top 64 features are kept, and the others are set to zero. This enforces sparsity and makes the resulting features easier to interpret.

- Embeddings are expanded into 16,384 possible features
- At most k = 64 features are active (non-zero) per residue
- Each feature is encouraged to be approximately monosemantic (capturing one interpretable concept) through a large feature space combined with a sparsity constraint
- Activation magnitude reflects how strongly the learned SAE feature is active at a given position; by looking at each feature's activation in a database of well characterized proteins, we can hypothesize the meaning of the SAE feature
- Optional normalization (`normalize_features=True`) down-weights the activation of very common features

In this tutorial, you’ll learn how to:
- Extract SAE features from any protein using the ESMC Biohub API
- Rank features by both peak activation and prevalence across the sequence
- Identify features that activate locally (on specific residues) and those that active more broadly across the protein
- Retrieve human-readable feature descriptions
- Visualize feature activations along sequences and map them onto 3D structures


# Setup

In [2]:
# If you are working in colab, uncomment these lines to install dependencies

!pip install -q "esm @ git+https://github.com/yashizhang/esm.git@main"
!pip install -q py3Dmol matplotlib requests numpy 

In [3]:
from functools import lru_cache
from getpass import getpass
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import py3Dmol
import requests

from esm.sdk.api import ESMProtein, LogitsConfig, SAEConfig
from esm.sdk.forge import ESMCForgeInferenceClient

First, we need to set up the Biohub client. Generate an API key on the [Biohub platform](https://biohub.ai) and add it to your account. This API key manages your access to credits and tokens, and the term API key/token is often used interchangeably within documentation.

Please note that your API key is like a password for your account and you should take care to protect it. For this reason it is recommended to frequently create a new API key and delete old, unused ones. It is also recommended to paste the API key directly into an environment variable or use a utility like `getpass` as shown in the example below so keys are not accidentally shared or checked into code repositories.

In [ ]:
token = getpass("Token from Biohub:")

# Initialize ESMC client with SAE support
model = ESMCForgeInferenceClient(
    model="esmc-6b-2024-12", url="https://biohub.ai", token=token
)

# Extract SAE Features from ATP Synthase


To make this concrete, we’ll analyze a well-characterized protein: **ATP synthase**.

ATP synthase is a molecular motor responsible for producing ATP, the primary energy currency of the cell. The F₁ subunit contains conserved nucleotide-binding motifs (including Walker A/B motifs) and well-defined structural domains. ATP synthase is a useful example because it contains well-studied **nucleotide-binding motifs**, including the Walker A and Walker B motifs that bind ATP.

These motifs typically appear as **localized activation peaks** in feature maps because they involve short conserved residue patterns.

At the same time, the protein also contains larger **structural domains** that may produce broader feature activation patterns.

We will use the structure from **PDB entry 2XND**, chain A, which corresponds to the alpha subunit of mitochondiral ATP synthase F. This protein contains:

- Conserved nucleotide-binding regions  
- Repeated structural motifs  
- Distinct secondary structure elements  
- Well-annotated functional sites  

This makes it a good example for observing both:
- **Localized features** (motif-like activations)  
- **Broad features** (domain- or structure-level activations)


To extract SAE features, we first need the amino acid sequence. Rather than hard-coding it, we’ll retrieve it directly from the RCSB PDB FASTA endpoint to ensure reproducibility.

In [ ]:
VALID_AAS = set("ACDEFGHIKLMNPQRSTVWY")


def validate_protein_sequence(sequence: str) -> str:
    """Validate and normalize a protein sequence."""
    sequence = sequence.strip().upper()
    invalid = sorted(set(sequence) - VALID_AAS)
    if invalid:
        raise ValueError(f"Invalid amino acid characters found: {invalid}")
    return sequence

In [ ]:
def fetch_pdb_sequence(pdb_id: str, chain_id: str) -> str:
    url = f"https://www.rcsb.org/fasta/entry/{pdb_id}"
    response = requests.get(url, timeout=10)
    if response.status_code != 200:
        raise ValueError(f"Failed to fetch PDB {pdb_id} (HTTP {response.status_code})")

    lines = response.text.strip().split("\n")
    sequences = {}
    current_chains = []
    current_seq = []

    for line in lines:
        if line.startswith(">"):
            if current_chains:
                seq = "".join(current_seq)
                for c in current_chains:
                    sequences[c] = seq
            # Parse both "Chain A" and "Chains A, B, C"
            parts = line.split("|")
            current_chains = []
            if len(parts) > 1:
                chain_part = parts[1].strip()
                if chain_part.startswith("Chain"):
                    # Remove "Chain" or "Chains" prefix and split by comma
                    chain_str = (
                        chain_part.replace("Chains", "").replace("Chain", "").strip()
                    )
                    current_chains = [c.strip() for c in chain_str.split(",")]
            current_seq = []
        else:
            current_seq.append(line.strip())

    if current_chains:
        seq = "".join(current_seq)
        for c in current_chains:
            sequences[c] = seq

    if chain_id not in sequences:
        available = list(sequences.keys())
        raise ValueError(f"Chain '{chain_id}' not found. Available: {available}")

    return validate_protein_sequence(sequences[chain_id])

In [ ]:
atp_synthase_sequence = fetch_pdb_sequence("2xnd", "A")
print(f"ATP synthase sequence ({len(atp_synthase_sequence)} residues):")
print(atp_synthase_sequence)

Now we'll extract SAE features using the Biohub API. We use:
- **Model**: `esmc-6b-2024-12` (6 billion parameter ESMC model)
- **SAE**: `esmc-6b-2024-12-sae-layer60-k64-codebook16384` (k=64 means top 64 features per position)
- **Normalization**: `normalize_features=True` applies TF-IDF normalization to down-weight common features

In [ ]:
# Create protein object
protein = ESMProtein(sequence=atp_synthase_sequence)

# Encode protein
protein_tensor = model.encode(protein)

# Get SAE features
sae_model_name = "esmc-6b-2024-12-sae-layer60-k64-codebook16384"
output = model.logits(
    protein_tensor,
    config=LogitsConfig(
        sae_config=SAEConfig(
            model=sae_model_name,
            normalize_features=True,  # TF-IDF normalization
        )
    ),
    return_bytes=False,
)

ESMC processes your protein sequence by first adding two special tokens: a beginning-of-sequence (BOS) token at the start and an end-of-sequence (EOS) token at the end. These help the model understand where a sequence starts and stops, but they don't correspond to any actual amino acid residue. Because of this, the feature matrix returned by the SAE has two extra rows, one for BOS and one for EOS. We remove them here so that each row in features maps cleanly to one residue in your sequence. This ensures that when we identify high-activating features later, the positions align correctly with the amino acids you care about.

In [ ]:
# Extract features (shape: [seq_len, codebook_size])
features = output.sae_outputs[sae_model_name].to_dense().numpy()
features = features[1:-1]  # Remove BOS/EOS tokens

print(f"Feature matrix shape: {features.shape}")
print(f"Sequence length: {len(atp_synthase_sequence)}")
print(f"Number of possible features: {features.shape[1]}")
print(
    f"\nSparsity: {100 * (features == 0).sum() / features.size:.1f}% of entries are zero"
)

In [ ]:
active_per_residue = (features > 0).sum(axis=1)

print(f"Average active features per residue: {active_per_residue.mean():.1f}")
print(f"Min active features at any residue: {active_per_residue.min()}")
print(f"Max active features at any residue: {active_per_residue.max()}")

The feature matrix has shape `[sequence_length, 16384]`. Each position in the protein has a vector of 16,384 possible features, but most are zero (sparse). Only the top 64 features per position are non-zero due to the top-k constraint in the SAE.

# Find Top Features

Which features are most active in ATP synthase? Let's find the top features by taking the maximum activation across all positions:

In [ ]:
# Summarize features in multiple ways
max_activations = features.max(axis=0)
prevalence = (features > 0.01).sum(axis=0)

mean_nonzero = np.array(
    [
        features[:, i][features[:, i] > 0.01].mean()
        if (features[:, i] > 0.01).any()
        else 0.0
        for i in range(features.shape[1])
    ]
)

top_k = 10
top_by_max = np.argsort(max_activations)[::-1][:top_k]
top_by_prevalence = np.argsort(prevalence)[::-1][:top_k]

print("Top 10 features by maximum activation:\n")
for idx, feature_id in enumerate(top_by_max, 1):
    print(
        f"{idx:2d}. Feature {feature_id:5d}: "
        f"max={max_activations[feature_id]:.3f}, "
        f"prevalence={prevalence[feature_id]}, "
        f"mean_nonzero={mean_nonzero[feature_id]:.3f}"
    )

print("\nTop 10 features by prevalence:\n")
for idx, feature_id in enumerate(top_by_prevalence, 1):
    print(
        f"{idx:2d}. Feature {feature_id:5d}: "
        f"max={max_activations[feature_id]:.3f}, "
        f"prevalence={prevalence[feature_id]}, "
        f"mean_nonzero={mean_nonzero[feature_id]:.3f}"
    )

Features with high prevalence (i.e., active across many residues) may reflect broad properties such as structural domains, fold-level patterns, or protein family signals. For example, Feature 2642 shows high prevalence in this protein, making it a candidate for such a global signal, although this requires further validation.

# Get Feature Descriptions

What do these features mean biologically? We've generated descriptions for each feature using an agent-based workflow that analyzes activations across millions of proteins. Let's fetch the descriptions:

*Note: These descriptions are automatically generated hypotheses based on large-scale activation patterns across many proteins. They should be treated as suggestive interpretations, not definitive annotations.

In [ ]:
# NOTE: This is an alpha API and likely to change!
@lru_cache(maxsize=16384)
def get_feature_info(feature_idx: int) -> dict[str, Any]:
    """Fetch metadata for a single SAE feature from the ESM Atlas API.

    Parameters
    ----------
    feature_idx : int
        Index of the SAE feature.

    Returns
    -------
    dict
        Parsed JSON response. Contains ``description`` and activation
        statistics. Check the biohub api documentation for details on the
        response format.
    """
    base_url = "https://biohub.ai"
    url = f"{base_url}/esm/protein/api/v1alpha1/features/{feature_idx}"
    response = requests.get(url)
    response.raise_for_status()
    return response.json()


# Get descriptions for top 5 features
print("Top 5 feature descriptions (ranked by max activation):\n")
for idx, feature_id in enumerate(top_by_max[:5], 1):
    info = get_feature_info(feature_id)
    top_uniref = info["top_100_uniref_ids"][0]
    print(f"{idx}. Feature {feature_id}")
    print(f"   description: {info.get('description', 'No description available')}")
    print(f"   top uniref:  {top_uniref}")

## Plot and Compare Features on the Sequence

Rather than assuming that particular feature IDs always correspond to fixed biological concepts, we'll inspect a few highly activating features in this protein and treat their descriptions as hypotheses about what the model may be detecting.

In [ ]:
# Note: The following cell defines visualization helpers - you can skip reading it!


def plot_feature_on_sequence(
    sequence: str, features: np.ndarray, feature_id: int, description: str = None
):
    """Plot a single feature's activation along the sequence."""
    activations = features[:, feature_id]
    positions = np.arange(1, len(sequence) + 1)

    fig, ax = plt.subplots(figsize=(14, 3))
    ax.plot(positions, activations, linewidth=1.5)

    ax.set_xlabel("Residue position")
    ax.set_ylabel("Activation")

    title = f"Feature {feature_id}"
    if description:
        desc_short = (
            description[:100] + "..." if len(description) > 100 else description
        )
        title += f": {desc_short}"
    ax.set_title(title)

    if (activations > 0).any():
        top_positions = np.argsort(activations)[::-1][:5]
        for pos in top_positions:
            if activations[pos] > 0:
                ax.annotate(
                    f"{sequence[pos]}{pos+1}",
                    (pos + 1, activations[pos]),
                    textcoords="offset points",
                    xytext=(0, 6),
                    ha="center",
                    fontsize=8,
                )

    plt.tight_layout()
    plt.show()

    print("Top activating positions:")
    top_positions = np.argsort(activations)[::-1][:5]
    for pos in top_positions:
        if activations[pos] > 0:
            print(f"  Position {pos+1} ({sequence[pos]}): {activations[pos]:.3f}")
    print()

In [ ]:
# Note: Comparison visualization helper - you can skip reading this!


def plot_multiple_features(sequence, features, feature_ids, descriptions=None):
    """Plot multiple features side-by-side."""
    n_features = len(feature_ids)
    fig, axes = plt.subplots(n_features, 1, figsize=(14, 2 * n_features), sharex=True)
    if n_features == 1:
        axes = [axes]

    for idx, (ax, feat_id) in enumerate(zip(axes, feature_ids)):
        activations = features[:, feat_id]
        positions = np.arange(1, len(sequence) + 1)

        ax.bar(positions, activations, width=1.0, color="steelblue", alpha=0.7)

        title = f"Feature {feat_id}"
        if descriptions and idx < len(descriptions):
            desc_short = (
                descriptions[idx][:60] + "..."
                if len(descriptions[idx]) > 60
                else descriptions[idx]
            )
            title += f": {desc_short}"
        ax.set_ylabel("Activation", fontsize=9)
        ax.set_title(title, fontsize=10)

    axes[-1].set_xlabel("Residue Position")
    plt.tight_layout()
    plt.show()

In [ ]:
# Compare the top 3 features
compare_features = top_by_max[:3]
compare_descriptions = [get_feature_info(f)["description"] for f in compare_features]

plot_multiple_features(
    atp_synthase_sequence, features, compare_features, compare_descriptions
)

Comparing the three features side-by-side reveals distinct activation patterns:

- Feature 9166 shows broad activation across the C-terminal half of the sequence (~300–500). This pattern suggests sensitivity to a large structural region or domain-level property.

- Feature 11587 exhibits a broad, bell-shaped activation centered in the middle of the sequence (~150–400). This type of pattern is consistent with features that track extended structural regions or conserved domains.

- Feature 10425 displays a sharp, localized peak around positions 263–267, with near-zero activation elsewhere. This is characteristic of features that detect short sequence motifs or localized functional sites.

The automatically generated descriptions for these features suggest associations with nucleotide-binding and P-loop NTPase-related patterns. While this is consistent with known biology of ATP synthase, these assignments should be treated as hypotheses rather than confirmed annotations.

To validate such hypotheses, one could compare activation peaks with annotated motifs (e.g., Walker A/B motifs), conserved residues, or experimentally characterized functional sites.





# 3D Structure Visualization

For proteins with known structures, feature activations can be mapped onto 3D coordinates. Here, we visualize SAE feature activations on the crystal structure of ATP synthase (PDB: 2XND).

Projecting activations onto structure can help reveal whether highly activating regions cluster in spatially meaningful locations, such as active sites, ligand-binding pockets, or domain interfaces.

However, this analysis requires careful alignment between the sequence used for feature extraction and the residues present in the structure. Differences such as missing residues, numbering offsets, or sequence variants can lead to misleading visualizations if not properly accounted for.

In [ ]:
# Note: 3D visualization helper - you can skip reading this!


def interpolate_color(value, min_val=0.0, max_val=1.0):
    """Interpolate between white (0) and red (1)."""
    if max_val == min_val:
        return "white"
    t = (value - min_val) / (max_val - min_val)
    t = max(0, min(1, t))  # Clamp to [0, 1]

    # White to red gradient
    r = int(255)
    g = int(255 * (1 - t))
    b = int(255 * (1 - t))
    return f"#{r:02x}{g:02x}{b:02x}"


def visualize_feature_on_structure(
    pdb_id, chain, features, feature_id, width=800, height=600
):
    view = py3Dmol.view(width=width, height=height)
    pdb_url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    response = requests.get(pdb_url)
    if response.status_code != 200:
        raise ValueError(f"Failed to fetch PDB {pdb_id}")
    view.addModel(response.text, "pdb")

    # Hide everything first
    view.setStyle({}, {"cartoon": {"hidden": True}})
    view.setStyle({"hetflag": True}, {"stick": {"hidden": True}})

    # Color chain of interest by feature activation
    activations = features[:, feature_id]
    max_act = activations.max()

    for i, act in enumerate(activations):
        color = interpolate_color(act, 0, max_act)
        view.setStyle({"chain": chain, "resi": i + 1}, {"cartoon": {"color": color}})

    view.zoomTo({"chain": chain})
    return view

In [ ]:
for feature_id in compare_features:
    info = get_feature_info(feature_id)
    print(f"Feature {feature_id}:")
    print(f"  description: {info['description']}")
    print(f"  top uniref:  {info['top_100_uniref_ids'][0]}")
    view = visualize_feature_on_structure("2xnd", "A", features, feature_id)
    view.show()

The 3D visualizations show that:

- Feature 9166 activation is concentrated in one region of the structure, consistent with a domain-level signal.

- Feature 11587 spans a broader structural region, suggesting sensitivity to a larger structural unit.

- Feature 10425 is localized to a narrow region, consistent with the motif-like activation observed in the sequence.

# Try Your Own Protein!

Now it's your turn! Pick any protein with a known structure in the PDB and run the same analysis. Just change the `pdb_id` and `chain_id` below.

Some suggestions to get started:

- **`1lyz`, chain A** — lysozyme (antimicrobial enzyme, ~130 residues)
- **`1zni`, chain A** — insulin (hormone signaling, small)
- **`1ubq`, chain A** — ubiquitin (regulatory protein, very small)
- **`4hhb`, chain A** — hemoglobin alpha subunit (oxygen transport)

Or browse [RCSB PDB](https://www.rcsb.org/) for a protein of interest.

In [ ]:
def extract_sae_features(sequence: str, model, sae_model_name: str):
    """Encode a sequence and extract SAE feature matrix."""
    sequence = validate_protein_sequence(sequence)
    protein = ESMProtein(sequence=sequence)
    protein_tensor = model.encode(protein)

    output = model.logits(
        protein_tensor,
        config=LogitsConfig(
            sae_config=SAEConfig(model=sae_model_name, normalize_features=True)
        ),
        return_bytes=False,
    )

    features = output.sae_outputs[sae_model_name].to_dense().numpy()[1:-1]
    return sequence, features

In [ ]:
# Change these to analyze a different protein
your_pdb_id = "1lyz"
your_chain_id = "A"

# Fetch the sequence and extract SAE features
your_sequence = fetch_pdb_sequence(your_pdb_id, your_chain_id)
your_sequence, your_features = extract_sae_features(
    your_sequence, model, sae_model_name
)

your_max_activations = your_features.max(axis=0)
your_prevalence = (your_features > 0).sum(axis=0)
your_top_indices = np.argsort(your_max_activations)[::-1][:5]

print(
    f"Analyzing {your_pdb_id.upper()} chain {your_chain_id} ({len(your_sequence)} residues)"
)
print("\nTop 5 features in your protein:\n")
for idx, feature_id in enumerate(your_top_indices, 1):
    info = get_feature_info(feature_id)
    print(f"{idx}. Feature {feature_id}")
    print(f"   max activation: {your_max_activations[feature_id]:.3f}")
    print(f"   prevalence:     {your_prevalence[feature_id]}")
    print(f"   description:    {info['description']}")
    print(f"   top uniref:     {info['top_100_uniref_ids'][0]}\n")

In [ ]:
# Visualize top 3 features
your_descriptions = [get_feature_info(f)["description"] for f in your_top_indices[:3]]
plot_multiple_features(
    your_sequence, your_features, your_top_indices[:3], your_descriptions
)

## Visualize on 3D structure

Now let's map these features onto the 3D structure, just like we did for ATP synthase.

In [ ]:
# Visualize top 3 features on the 3D structure
for feature_id in your_top_indices[:3]:
    info = get_feature_info(feature_id)
    print(f"Feature {feature_id}:")
    print(f"  description: {info['description']}")
    print(f"  top uniref:  {info['top_100_uniref_ids'][0]}")
    view = visualize_feature_on_structure(
        your_pdb_id, your_chain_id, your_features, feature_id
    )
    view.show()

# Understanding Your Results

### Interpreting Feature Activations

SAE features can reflect biological patterns at several scales.

**Localized activation**
If a feature activates strongly in a narrow region of the sequence, it may correspond to:

- catalytic residues
- ligand-binding motifs
- conserved sequence motifs
- post-translational modification sites

These appear as sharp peaks in activation plots.

**Broad activation**
If a feature activates across a larger region, it may represent:

- structural domains
- secondary structure elements
- protein family identity
- large binding interfaces

**Global activation**
Features that activate across most of the sequence often reflect:

- overall fold class
- disorder vs structured proteins
- evolutionary or taxonomic signals

## Tips for Analysis

1. **Rank features in multiple ways.**  
   Use maximum activation to find motif-like features, and prevalence (number of active residues) to identify global structural or family-level features.

2. **Inspect activation patterns, not just magnitudes.**  
   Sharp peaks often indicate localized motifs. Broad activation may reflect global structural or evolutionary signals.

3. **Compare related proteins.**  
   Conserved feature activations across homologs often correspond to conserved functional or structural elements.

4. **Use 3D visualization carefully.**  
   Ensure residue numbering aligns between the extracted sequence and the structure file before interpreting spatial clustering.

5. **Treat descriptions as hypotheses.**  
   Feature annotations are automatically generated from large-scale activation patterns and may be incomplete or incorrect. Use them as starting points for investigation, not definitive labels.

## Limitations

- SAE features are interpretability signals, not experimentally validated annotations.
- High activation means the model strongly associates a concept with a region, not necessarily that the protein truly has that function.
- Feature descriptions are generated automatically and may be incomplete or incorrect for rare biology.
- Feature rankings depend on the aggregation (max vs prevalence) and normalization settings.
- Mapping features onto 3D structures requires careful residue alignment between the extracted sequence and the structure file.

# Next Steps

Now that you understand SAE features, you can:

**Search for similar proteins**: Use feature vectors to find proteins with similar properties in large databases. Proteins with similar feature profiles often have related functions.

**Engineer new proteins**: Design sequences with specific feature combinations. For example, you could engineer a protein to have features associated with both thermostability and catalytic activity.

**Discover novel biology**: Find features that activate together in unexpected ways. Co-occurring features might reveal functional relationships that aren't obvious from sequence or structure alone.

**Understand model behavior**: SAE features help explain what ESM has learned about proteins. This can guide better prompting strategies for protein generation with ESM3.

Check out the other tutorials [here](https://github.com/yashizhang/esm/tree/main/cookbook/tutorials).
